In [ ]:
import serial
import time
import re

# Function to send AT command
def send_command(command):
    ser.write((command + '\r\n').encode())
    time.sleep(1)
    response = ser.read(ser.inWaiting()).decode()
    return response

# Function to connect to MQTT broker
def connect_to_mqtt(host, port, user, password):
    command = 'AT+CGATT=1'
    send_command(command)
    command = 'AT+CGDCONT=1,"IP","CMNET"'
    send_command(command)
    command = 'AT+CGACT=1,1'
    send_command(command)
    # Connect to MQTT broker using A9G board
    command = 'AT+MQTTCONN="{}",{},12345,120,0,"{}","{}"'.format(host, port, user, password)
    send_command(command)

# Function to subscribe to MQTT topic
def subscribe_to_topic(topic):
    command = 'AT+MQTTSUB="{}",1,0'.format(topic)
    send_command(command)

# Main code
if __name__ == "__main__":
    # Open serial connection to the A9G module
    ser = serial.Serial('COM5', 115200, timeout=1)

    # Wait for the module to initialize
    time.sleep(2)

    try:
        # MQTT broker details
        mqtt_host = "0.tcp.in.ngrok.io"
        mqtt_port = "19731"
        mqtt_user = ""
        mqtt_password = ""
        topic_to_subscribe = "test_topic"

        # Connect to MQTT broker using A9G board
        connect_to_mqtt(mqtt_host, mqtt_port, mqtt_user, mqtt_password)

        # Subscribe to MQTT topic
        subscribe_to_topic(topic_to_subscribe)
        print("Subscribed to topic:", topic_to_subscribe)

        # Loop to receive MQTT messages
        while True:
            
            if ser.inWaiting() > 0:
                
                response = ser.read(ser.inWaiting()).decode()
                print(response)
                if "+MQTTPUBLISH:" in response:
                    print("here")
                    # Extract message from response
                    # match = re.search(r'\+MQTTPUBLISH: (\d+),"(.*)",(.+)', response)
                    # Assuming response is the string containing the received MQTT PUBLISH message
                    match = re.match(r'\+MQTTPUBLISH:(\d+),"(.+)",(\d+),(.+)', response)
                    print(match)
                    if match:
                        qos = match.group(1)
                        topic = match.group(2)
                        message_length = int(match.group(3))
                        message_content = match.group(4)
                    
                        print("Received message:")
                        print("  QoS:", qos)
                        print("  Topic:", topic)
                        print("  Message Length:", message_length)
                        print("  Message Content:", message_content)
            time.sleep(1)

    except KeyboardInterrupt:
        print("Exiting program.")

    # Close serial connection
    ser.close()


Subscribed to topic: test_topic

OK

+MQTTPUBLISH:1,test_topic,6,hello2

here
None
